# Traditional ML Synthetic Options Backtest

This notebook compares the current long-only `raw_pct6` stock strategy against synthetic constant-maturity call-option proxies.

Instrument variants:
- equity
- ATM call proxy
- OTM call proxy
- DITM call proxy

Strategy variants:
- `top_k = 5`
- `top_k = 10`
- `top_k = 20`

Synthetic approximation assumptions:
- long-only call proxies
- constant-maturity option index, rolled continuously each day
- Black-Scholes pricing with realized-volatility proxy
- ATM = strike multiplier `1.00`
- OTM = strike multiplier `1.05`
- DITM = strike multiplier `0.90`
- tenor fixed at `30` trading days

This is a research approximation, not a broker-realistic options backtest.


In [1]:
import gc
import json
import math
import os
import pickle
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import django
from django.apps import apps

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
if not apps.ready:
    django.setup()

from fmp.workflows import run_scoring_data_refresh_from_fmp
from fmp.models import Symbol
from features.feature_builders import build_price_technical_features
from features.analyst_estimates_features import build_analyst_estimates_features
from features.balance_sheet_features import build_balance_sheet_features
from features.balance_sheet_growth_features import build_balance_sheet_growth_features
from features.cash_flow_features import build_cash_flow_features
from features.cash_flow_growth_features import build_cash_flow_growth_features
from features.earnings_features import build_earnings_features
from features.financial_growth_features import build_financial_growth_features
from features.grades_historical_features import build_grades_historical_features
from features.income_statement_features import build_income_statement_features
from features.income_statement_growth_features import build_income_statement_growth_features
from features.insider_trading_features import build_insider_trading_features
from features.key_metrics_features import build_key_metrics_features
from features.ratios_features import build_ratios_features
from features.ratings_historical_features import build_ratings_historical_features
from features.section_utils import clear_section_record_cache, prime_section_record_cache
from features.views import _load_adjusted_prices
from pipeline.api import build_macro_dataframe, build_label_dataframe
from data.preparation import MLDatasetConfig, prepare_ml_dataset
from ml.base import FitSpec
from ml.frameworks.sklearn import SklearnRFClassifier, SklearnRFRegressor
from ml.raw_stack import train_ae
from backtest.raw_stack import ProbabilityColumnConfig, enrich_scored_panel, make_backtest_panel
from backtest.latest import make_autoencoder_familiarity_predictor, run_panel_prediction_custom
from pipeline.universe_selection import DEFAULT_US_EXCHANGES, resolve_symbol_universe
from trading.live_trade import resolve_fmp_api_key

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def current_memory_mb():
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        try:
            import resource
            usage = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
            # macOS reports ru_maxrss in bytes; Linux usually reports KB.
            return usage / (1024 ** 2) if usage > 10_000_000 else usage / 1024.0
        except Exception:
            return float("nan")


def log_memory(label, *, rows=None, cols=None, collect=False):
    if collect:
        gc.collect()
    rss_mb = current_memory_mb()
    details = []
    if rows is not None:
        details.append(f"rows={int(rows):,}")
    if cols is not None:
        details.append(f"cols={int(cols):,}")
    detail_text = " | " + " | ".join(details) if details else ""
    print(f"[memory] {label}: rss={rss_mb:,.1f} MB{detail_text}")
    return rss_mb



def _build_technical_dataframe_from_django(*, symbols, start_date=None, end_date=None):
    start_ts = pd.Timestamp(start_date) if start_date is not None else None
    end_ts = pd.Timestamp(end_date) if end_date is not None else None
    frames = []
    feature_cols = []

    for sym in symbols:
        code = str(sym).strip().upper()
        if not code:
            continue

        symbol_obj = Symbol.objects.filter(symbol__iexact=code).only("id", "symbol").first()
        if symbol_obj is None:
            continue

        df_prices = _load_adjusted_prices(
            symbol_obj,
            start_ts.date() if start_ts is not None else None,
            end_ts.date() if end_ts is not None else None,
        )
        if df_prices.empty:
            continue

        built = build_price_technical_features(code, df_prices)
        if built.df.empty:
            continue

        px = df_prices[["open", "high", "low", "close", "volume"]].copy()
        px["symbol"] = code
        px = px.reset_index().set_index(["date", "symbol"]).sort_index()

        panel = px.join(built.df[built.feature_cols], how="left")
        frames.append(panel)

        for col in built.feature_cols:
            if col not in feature_cols:
                feature_cols.append(col)

    if not frames:
        empty_index = pd.MultiIndex(levels=[[], []], codes=[[], []], names=["date", "symbol"])
        return pd.DataFrame(index=empty_index), feature_cols

    technical_df = pd.concat(frames, axis=0).sort_index()
    if technical_df.index.has_duplicates:
        technical_df = technical_df[~technical_df.index.duplicated(keep="last")]
    return technical_df, feature_cols


def _target_index_for_symbol(target_index, symbol):
    code = str(symbol).strip().upper()
    mask = target_index.get_level_values("symbol").astype(str).str.upper() == code
    dates = pd.DatetimeIndex(pd.to_datetime(target_index.get_level_values("date")[mask])).normalize()
    return pd.MultiIndex.from_arrays([dates, [code] * len(dates)], names=["date", "symbol"])


def _price_frame_for_symbol(price_panel, symbol):
    code = str(symbol).strip().upper()
    try:
        return price_panel.xs(code, level="symbol").copy()
    except Exception:
        return pd.DataFrame()


def _build_fmp_endpoint_feature_families(*, symbols, target_index, price_panel, filing_lag_days=45):
    endpoint_builders = {
        "key_metrics": lambda symbol_obj, idx, px, market_cap, valuation_context: build_key_metrics_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "ratios": lambda symbol_obj, idx, px, market_cap, valuation_context: build_ratios_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "income_statement": lambda symbol_obj, idx, px, market_cap, valuation_context: build_income_statement_features(symbol_obj, idx, df_prices=px, filing_lag_days=filing_lag_days),
        "income_statement_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_income_statement_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "balance_sheet": lambda symbol_obj, idx, px, market_cap, valuation_context: build_balance_sheet_features(symbol_obj, idx, df_prices=px, market_cap=market_cap, filing_lag_days=filing_lag_days),
        "balance_sheet_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_balance_sheet_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "cash_flow": lambda symbol_obj, idx, px, market_cap, valuation_context: build_cash_flow_features(symbol_obj, idx, df_prices=px, market_cap=market_cap, filing_lag_days=filing_lag_days),
        "cash_flow_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_cash_flow_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "financial_growth": lambda symbol_obj, idx, px, market_cap, valuation_context: build_financial_growth_features(symbol_obj, idx, valuation_frame=valuation_context, filing_lag_days=filing_lag_days),
        "earnings": lambda symbol_obj, idx, px, market_cap, valuation_context: build_earnings_features(symbol_obj, idx),
        "analyst_estimates": lambda symbol_obj, idx, px, market_cap, valuation_context: build_analyst_estimates_features(symbol_obj, idx, df_prices=px, market_cap=market_cap),
        "ratings_historical": lambda symbol_obj, idx, px, market_cap, valuation_context: build_ratings_historical_features(symbol_obj, idx),
        "grades_historical": lambda symbol_obj, idx, px, market_cap, valuation_context: build_grades_historical_features(symbol_obj, idx),
        "insider_trading": lambda symbol_obj, idx, px, market_cap, valuation_context: build_insider_trading_features(symbol_obj, idx),
    }
    normalized_symbols = [str(s).strip().upper() for s in symbols if str(s).strip()]
    symbol_objs = {
        str(obj.symbol).strip().upper(): obj
        for obj in Symbol.objects.filter(symbol__in=normalized_symbols)
    }
    endpoint_frames = {name: [] for name in endpoint_builders}
    endpoint_cols = {name: [] for name in endpoint_builders}
    try:
        prime_section_record_cache(list(symbol_objs.values()), list(endpoint_builders.keys()))
        for code in normalized_symbols:
            symbol_obj = symbol_objs.get(code) or Symbol.objects.filter(symbol__iexact=code).first()
            if symbol_obj is None:
                continue
            symbol_index = _target_index_for_symbol(target_index, code)
            if len(symbol_index) == 0:
                continue
            symbol_prices = _price_frame_for_symbol(price_panel, code)
            symbol_market_cap = None
            valuation_context = pd.DataFrame(index=symbol_index)
            for endpoint_name, builder in endpoint_builders.items():
                built = builder(symbol_obj, symbol_index, symbol_prices, symbol_market_cap, valuation_context)
                active_cols = [c for c in built.feature_cols if c in built.df.columns and pd.api.types.is_numeric_dtype(built.df[c])]
                if endpoint_name == "key_metrics" and "km__marketcap" in built.df.columns:
                    symbol_market_cap = pd.to_numeric(built.df["km__marketcap"], errors="coerce")
                if endpoint_name in {"key_metrics", "ratios", "income_statement", "balance_sheet", "cash_flow"} and not built.df.empty:
                    valuation_context = pd.concat([valuation_context, built.df], axis=1)
                    if valuation_context.columns.has_duplicates:
                        valuation_context = valuation_context.loc[:, ~valuation_context.columns.duplicated(keep="last")]
                if not active_cols:
                    continue
                endpoint_frames[endpoint_name].append(built.df[active_cols].copy())
                for col in active_cols:
                    if col not in endpoint_cols[endpoint_name]:
                        endpoint_cols[endpoint_name].append(col)
    finally:
        clear_section_record_cache()

    family_frames = {}
    family_cols = {}
    for endpoint_name, frames in endpoint_frames.items():
        if not frames:
            continue
        frame = pd.concat(frames, axis=0).sort_index()
        if frame.index.has_duplicates:
            frame = frame[~frame.index.duplicated(keep="last")]
        cols = [c for c in endpoint_cols[endpoint_name] if c in frame.columns and pd.api.types.is_numeric_dtype(frame[c])]
        if cols:
            family_frames[endpoint_name] = frame[cols].copy()
            family_cols[endpoint_name] = cols
    return family_frames, family_cols


def summarize_curve(returns, years, mode):
    returns = pd.Series(returns).fillna(0.0)
    equity = (1.0 + returns).cumprod()
    total_return_pct = float((equity.iloc[-1] - 1.0) * 100.0) if len(equity) else np.nan
    sharpe = float((returns.mean() / returns.std(ddof=0)) * np.sqrt(252.0)) if len(returns) and returns.std(ddof=0) > 1e-12 else np.nan
    max_drawdown_pct = float((((equity / equity.cummax()) - 1.0).min()) * 100.0) if len(equity) else np.nan
    yearly_rows = []
    for yr in years:
        yret = returns.loc[(returns.index >= pd.Timestamp(f"{yr}-01-01")) & (returns.index <= pd.Timestamp(f"{yr}-12-31"))]
        yeq = (1.0 + yret).cumprod()
        yearly_rows.append(
            {
                "mode": str(mode),
                "test_year": int(yr),
                "total_return_pct": float((yeq.iloc[-1] - 1.0) * 100.0) if len(yeq) else np.nan,
                "sharpe": float((yret.mean() / yret.std(ddof=0)) * np.sqrt(252.0)) if len(yret) and yret.std(ddof=0) > 1e-12 else np.nan,
                "max_drawdown_pct": float((((yeq / yeq.cummax()) - 1.0).min()) * 100.0) if len(yeq) else np.nan,
            }
        )
    return {
        "total_return_pct": total_return_pct,
        "sharpe": sharpe,
        "max_drawdown_pct": max_drawdown_pct,
        "equity_curve": equity,
        "yearly_df": pd.DataFrame(yearly_rows),
    }


def _pivot_rule_panel(panel, col, *, symbols=None):
    working_symbols = sorted(panel.index.get_level_values("symbol").unique()) if symbols is None else list(symbols)
    return (
        panel[[col]]
        .reset_index()
        .pivot(index="date", columns="symbol", values=col)
        .reindex(columns=working_symbols)
        .sort_index()
    )


def _prepare_capacity_rule_inputs(panel, score_col, component_cols, price_col):
    symbols = sorted(panel.index.get_level_values("symbol").unique())
    score = _pivot_rule_panel(panel, score_col, symbols=symbols).shift(1)
    prob_buy = _pivot_rule_panel(panel, "prob_buy", symbols=symbols).shift(1)
    prob_short = _pivot_rule_panel(panel, "prob_short", symbols=symbols).shift(1)
    close = _pivot_rule_panel(panel, price_col, symbols=symbols)
    common_dates = score.index.intersection(prob_buy.index).intersection(prob_short.index).intersection(close.index)
    score = score.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    prob_buy = prob_buy.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    prob_short = prob_short.loc[common_dates].replace([np.inf, -np.inf], np.nan)
    close = close.loc[common_dates].replace([np.inf, -np.inf], np.nan).ffill().fillna(0.0)
    component_frames = {}
    for col in component_cols:
        component = _pivot_rule_panel(panel, col, symbols=symbols).shift(1).reindex(index=common_dates, columns=symbols)
        component_frames[str(col)] = component.replace([np.inf, -np.inf], np.nan)
    return {
        "symbols": symbols,
        "common_dates": common_dates,
        "score": score,
        "prob_buy": prob_buy,
        "prob_short": prob_short,
        "close": close,
        "component_cols": [str(col) for col in component_cols],
        "component_frames": component_frames,
    }


def _build_entry_ok_matrix(inputs, component_threshold):
    score = inputs["score"]
    close = inputs["close"]
    entry_ok = (score.notna() & np.isfinite(score) & close.gt(0.0)).fillna(False)
    for col in inputs["component_cols"]:
        component = inputs["component_frames"][col]
        component_valid = component.notna() & np.isfinite(component)
        entry_ok &= (component.gt(float(component_threshold)) & component_valid).fillna(False)
    return entry_ok


def _run_capacity_limited_long_only_rule(*, panel, score_col, component_cols, component_threshold, price_col, top_k=None):
    inputs = _prepare_capacity_rule_inputs(panel, score_col, component_cols, price_col)
    symbols = inputs["symbols"]
    common_dates = inputs["common_dates"]
    score = inputs["score"]
    prob_buy = inputs["prob_buy"]
    prob_short = inputs["prob_short"]
    close = inputs["close"]

    entry_ok = _build_entry_ok_matrix(inputs, component_threshold)

    held_idx = set()
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        prob_buy_t = prob_buy.loc[dt]
        prob_short_t = prob_short.loc[dt]
        classifier_short = (prob_short_t > prob_buy_t).fillna(False)

        exit_idx = sorted(
            idx
            for idx in held_idx
            if bool(classifier_short.iloc[idx])
        )
        if exit_idx:
            held_idx -= set(exit_idx)

        slots_left = None if top_k is None else max(0, int(top_k) - len(held_idx))
        if slots_left != 0:
            price_ok_t = close.loc[dt].gt(0.0)
            candidate_mask = entry_ok.loc[dt] & price_ok_t & (~classifier_short)
            ranked = score.loc[dt][candidate_mask].sort_values(ascending=False, kind="stable")
            enter_idx = []
            for sym in ranked.index:
                idx = symbol_to_idx[str(sym)]
                if idx in held_idx:
                    continue
                enter_idx.append(idx)
                if slots_left is not None and len(enter_idx) >= slots_left:
                    break
            if enter_idx:
                held_idx |= set(enter_idx)

        if held_idx:
            position_by_day.loc[dt, [symbols[idx] for idx in sorted(held_idx)]] = 1

    return {
        "positions": position_by_day,
        "score": score,
        "close": close,
    }


def run_top_k_long_only_score_rule(*, panel, score_col, component_cols, component_threshold, price_col, top_k, rebalance_freq=None):
    _ = rebalance_freq
    return _run_capacity_limited_long_only_rule(
        panel=panel,
        score_col=score_col,
        component_cols=component_cols,
        component_threshold=component_threshold,
        price_col=price_col,
        top_k=int(top_k),
    )


def _run_capacity_limited_long_short_rule(
    *,
    panel,
    long_score_col,
    short_score_col,
    long_component_cols,
    short_component_cols,
    component_threshold,
    price_col,
    top_k=None,
):
    long_inputs = _prepare_capacity_rule_inputs(panel, long_score_col, long_component_cols, price_col)
    short_inputs = _prepare_capacity_rule_inputs(panel, short_score_col, short_component_cols, price_col)
    symbols = long_inputs["symbols"]
    common_dates = long_inputs["common_dates"]
    close = long_inputs["close"]
    long_score = long_inputs["score"]
    short_score = short_inputs["score"]
    prob_buy = long_inputs["prob_buy"]
    prob_short = long_inputs["prob_short"]

    long_entry_ok = _build_entry_ok_matrix(long_inputs, component_threshold)
    short_entry_ok = _build_entry_ok_matrix(short_inputs, component_threshold)

    held_side_by_idx = {}
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        price_ok_t = close.loc[dt].gt(0.0)
        prob_buy_t = prob_buy.loc[dt]
        prob_short_t = prob_short.loc[dt]
        long_score_t = long_score.loc[dt]
        short_score_t = short_score.loc[dt]

        next_held = {}
        for idx, side in sorted(held_side_by_idx.items()):
            if side > 0:
                if bool(prob_short_t.iloc[idx] > prob_buy_t.iloc[idx]):
                    continue
            else:
                if bool(prob_buy_t.iloc[idx] > prob_short_t.iloc[idx]):
                    continue
            next_held[idx] = side
        held_side_by_idx = next_held

        capacity = None if top_k is None else max(0, int(top_k))
        slots_left = None if capacity is None else max(0, capacity - len(held_side_by_idx))
        if slots_left != 0:
            candidates = []
            for sym in symbols:
                idx = symbol_to_idx[str(sym)]
                if idx in held_side_by_idx or (not bool(price_ok_t.iloc[idx])):
                    continue
                long_ok = bool(long_entry_ok.loc[dt].iloc[idx]) and np.isfinite(long_score_t.iloc[idx])
                short_ok = bool(short_entry_ok.loc[dt].iloc[idx]) and np.isfinite(short_score_t.iloc[idx])
                if not long_ok and not short_ok:
                    continue
                if long_ok and short_ok:
                    long_value = float(long_score_t.iloc[idx])
                    short_value = float(short_score_t.iloc[idx])
                    if long_value >= short_value:
                        best_side, best_score = 1, long_value
                    else:
                        best_side, best_score = -1, short_value
                elif long_ok:
                    best_side, best_score = 1, float(long_score_t.iloc[idx])
                else:
                    best_side, best_score = -1, float(short_score_t.iloc[idx])
                candidates.append((best_score, str(sym), idx, best_side))

            for _score_value, _sym, idx, side in sorted(candidates, key=lambda row: (row[0], row[1]), reverse=True):
                held_side_by_idx[idx] = int(side)
                if slots_left is not None and len(held_side_by_idx) >= int(capacity):
                    break

        for idx, side in sorted(held_side_by_idx.items()):
            position_by_day.loc[dt, symbols[idx]] = int(side)

    return {
        "positions": position_by_day,
        "long_score": long_score,
        "short_score": short_score,
        "close": close,
    }


def run_top_k_long_short_score_rule(
    *,
    panel,
    long_score_col,
    short_score_col,
    long_component_cols,
    short_component_cols,
    component_threshold,
    price_col,
    top_k,
    rebalance_freq=None,
):
    _ = rebalance_freq
    return _run_capacity_limited_long_short_rule(
        panel=panel,
        long_score_col=long_score_col,
        short_score_col=short_score_col,
        long_component_cols=long_component_cols,
        short_component_cols=short_component_cols,
        component_threshold=component_threshold,
        price_col=price_col,
        top_k=int(top_k),
    )


def run_top_k_momentum_baseline(*, panel, price_col, top_k, lookback_days=21, rebalance_freq=None):
    _ = rebalance_freq
    symbols = sorted(panel.index.get_level_values("symbol").unique())
    close = _pivot_rule_panel(panel, price_col, symbols=symbols).replace([np.inf, -np.inf], np.nan).ffill()
    score = close.pct_change(int(lookback_days)).shift(1).replace([np.inf, -np.inf], np.nan)
    common_dates = score.index.intersection(close.index)
    score = score.loc[common_dates]
    close = close.loc[common_dates].fillna(0.0)
    held_side_by_idx = {}
    symbol_to_idx = {sym: idx for idx, sym in enumerate(symbols)}
    position_by_day = pd.DataFrame(0, index=common_dates, columns=symbols, dtype=int)

    for dt in common_dates:
        score_t = score.loc[dt]
        price_ok_t = close.loc[dt].gt(0.0)
        next_held = {}
        for idx, side in sorted(held_side_by_idx.items()):
            if not bool(price_ok_t.iloc[idx]) or (not np.isfinite(score_t.iloc[idx])):
                continue
            current_score = float(score_t.iloc[idx])
            if side > 0 and current_score <= 0.0:
                continue
            if side < 0 and current_score >= 0.0:
                continue
            next_held[idx] = side
        held_side_by_idx = next_held

        capacity = max(0, int(top_k))
        slots_left = max(0, capacity - len(held_side_by_idx))
        if slots_left != 0:
            candidates = []
            for sym in symbols:
                idx = symbol_to_idx[str(sym)]
                if idx in held_side_by_idx or (not bool(price_ok_t.iloc[idx])) or (not np.isfinite(score_t.iloc[idx])):
                    continue
                momentum_value = float(score_t.iloc[idx])
                if momentum_value > 0.0:
                    candidates.append((abs(momentum_value), str(sym), idx, 1))
                elif momentum_value < 0.0:
                    candidates.append((abs(momentum_value), str(sym), idx, -1))
            for _score_value, _sym, idx, side in sorted(candidates, key=lambda row: (row[0], row[1]), reverse=True):
                held_side_by_idx[idx] = int(side)
                if len(held_side_by_idx) >= capacity:
                    break

        for idx, side in sorted(held_side_by_idx.items()):
            position_by_day.loc[dt, symbols[idx]] = int(side)

    return {
        "positions": position_by_day,
        "score": score,
        "close": close,
    }


def resolve_component_cols(score_col):
    mapping = {
        "buy_score_mean_raw3": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "buy_score_mean_raw_pct6": [
            "prob_buy",
            "pred_rf_reg",
            "ae_familiarity",
            "prob_buy_pct",
            "pred_rf_reg_pct",
            "ae_familiarity_pct",
        ],
        "buy_score_pct_mean": ["prob_buy_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "buy_score_pct_product": ["prob_buy_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "buy_score_raw": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "buy_score": ["prob_buy", "pred_rf_reg", "ae_familiarity"],
        "short_score_mean_raw3": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "short_score_mean_raw_pct6": [
            "prob_short",
            "pred_rf_reg",
            "ae_familiarity",
            "prob_short_pct",
            "pred_rf_reg_pct",
            "ae_familiarity_pct",
        ],
        "short_score_pct_mean": ["prob_short_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "short_score_pct_product": ["prob_short_pct", "pred_rf_reg_pct", "ae_familiarity_pct"],
        "short_score_raw": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "short_score": ["prob_short", "pred_rf_reg", "ae_familiarity"],
        "__momentum_21d__": [],
    }
    return list(mapping.get(str(score_col), [str(score_col)]))


def resolve_short_score_col(score_col):
    mapping = {
        "buy_score_mean_raw3": "short_score_mean_raw3",
        "buy_score_mean_raw_pct6": "short_score_mean_raw_pct6",
        "buy_score_pct_mean": "short_score_pct_mean",
        "buy_score_pct_product": "short_score_pct_product",
        "buy_score_raw": "short_score_raw",
        "buy_score": "short_score",
    }
    key = str(score_col)
    if key in mapping:
        return str(mapping[key])
    if key.startswith("buy_"):
        return "short_" + key[len("buy_"):]
    raise KeyError(f"No short-score mapping configured for: {score_col}")


def _norm_cdf(x):
    x_arr = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(math.erf)(x_arr / np.sqrt(2.0)))


def build_realized_vol_panel(close_df, *, window=21, vol_floor=0.15, vol_cap=0.80):
    ret = close_df.pct_change()
    min_periods = min(int(window), max(5, int(window) // 2))
    vol = ret.rolling(int(window), min_periods=min_periods).std(ddof=0) * np.sqrt(252.0)
    vol = vol.shift(1)
    return vol.clip(lower=float(vol_floor), upper=float(vol_cap)).fillna(float(vol_floor))


def build_constant_maturity_call_price_panel(close_df, realized_vol_df, *, strike_multiplier, tenor_days=30, rate=0.0, iv_multiplier=1.0, premium_floor=0.25):
    spot = close_df.astype(float)
    sigma = realized_vol_df.astype(float) * float(iv_multiplier)
    tau = max(float(tenor_days) / 252.0, 1.0 / 252.0)
    m = float(strike_multiplier)
    sqrt_tau = math.sqrt(tau)

    log_term = math.log(1.0 / m)
    denom = (sigma * sqrt_tau).replace(0.0, np.nan)
    d1 = (log_term + (float(rate) + 0.5 * sigma * sigma) * tau) / denom
    d2 = d1 - sigma * sqrt_tau
    n1 = pd.DataFrame(_norm_cdf(d1.to_numpy(dtype=float)), index=d1.index, columns=d1.columns)
    n2 = pd.DataFrame(_norm_cdf(d2.to_numpy(dtype=float)), index=d2.index, columns=d2.columns)
    discount = math.exp(-float(rate) * tau)
    price = spot * n1 - (spot * m) * discount * n2
    intrinsic = (spot - (spot * m)).clip(lower=0.0)
    price = price.where(np.isfinite(price), intrinsic)
    return price.clip(lower=float(premium_floor))


def build_constant_maturity_put_price_panel(close_df, realized_vol_df, *, strike_multiplier, tenor_days=30, rate=0.0, iv_multiplier=1.0, premium_floor=0.25):
    spot = close_df.astype(float)
    sigma = realized_vol_df.astype(float) * float(iv_multiplier)
    tau = max(float(tenor_days) / 252.0, 1.0 / 252.0)
    m = float(strike_multiplier)
    sqrt_tau = math.sqrt(tau)

    log_term = math.log(1.0 / m)
    denom = (sigma * sqrt_tau).replace(0.0, np.nan)
    d1 = (log_term + (float(rate) + 0.5 * sigma * sigma) * tau) / denom
    d2 = d1 - sigma * sqrt_tau
    n1 = pd.DataFrame(_norm_cdf((-d1).to_numpy(dtype=float)), index=d1.index, columns=d1.columns)
    n2 = pd.DataFrame(_norm_cdf((-d2).to_numpy(dtype=float)), index=d2.index, columns=d2.columns)
    discount = math.exp(-float(rate) * tau)
    price = (spot * m) * discount * n2 - spot * n1
    intrinsic = ((spot * m) - spot).clip(lower=0.0)
    price = price.where(np.isfinite(price), intrinsic)
    return price.clip(lower=float(premium_floor))


def backtest_positions_with_directional_asset_returns(
    positions,
    long_asset_returns,
    *,
    short_asset_returns=None,
    initial_balance=100000.0,
    fee_bps=5.0,
    slippage_bps=5.0,
):
    positions = positions.astype(float).copy()
    long_asset_returns = long_asset_returns.reindex(index=positions.index, columns=positions.columns).fillna(0.0)
    if short_asset_returns is None:
        short_asset_returns = -long_asset_returns
    short_asset_returns = short_asset_returns.reindex(index=positions.index, columns=positions.columns).fillna(0.0)
    gross_weight_basis = positions.abs().sum(axis=1).replace(0.0, np.nan)
    weights = positions.div(gross_weight_basis, axis=0).fillna(0.0)
    lagged_weights = weights.shift(1).fillna(0.0)
    long_weights = lagged_weights.clip(lower=0.0)
    short_weights = (-lagged_weights.clip(upper=0.0))
    gross_returns = (long_weights * long_asset_returns + short_weights * short_asset_returns).sum(axis=1)
    turnover = 0.5 * weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost_rate = (float(fee_bps) + float(slippage_bps)) / 10000.0
    net_returns = gross_returns - turnover * cost_rate
    equity = (1.0 + net_returns).cumprod() * float(initial_balance)
    return {
        "weights": weights,
        "turnover": turnover,
        "returns": net_returns,
        "equity": equity,
    }



In [2]:
APP_CFG = {
    "dates": {
        "train_cutoff": "2020-12-31",
        "bt_start": "2021-01-01",
        "bt_end": pd.Timestamp.now(tz="America/Los_Angeles").date().isoformat(),
        "data_start": "2005-01-01",
        "score_start": "2020-01-01",
    },
    "universe": {
        "country": "US",
        "exchanges": list(DEFAULT_US_EXCHANGES),
        "min_market_cap": 10_000_000_000.0,
        "exclude_pooled_vehicles": True,
        "size": None,
    },
    "runtime": {
        "max_clf_train_rows": 250_000,
        "max_ae_train_rows": 100_000,
        "artifact_dir": os.path.abspath("artifacts/synthetic_options_raw_stack"),
    },
    "fmp_refresh": {
        "enabled": bool(pd.Timestamp.now(tz="America/Los_Angeles").hour >= 17),
        "refresh_symbol_sections_before_build": True,
        "refresh_macro_before_build": False,
        "mode": "scoring_ready",
        "existing_historical_sections_only": True,
        "max_symbols": None,
        "verbose": True,
    },
    "costs": {
        "fee_bps": 5.0,
        "slippage_bps": 5.0,
    },
    "labels": {
        "k_params": {"YE": [1, 2, 4, 8]},
        "use_sample_weight": True,
        "alpha": 4.0,
        "r_clip": 0.10,
        "horizon_balance": True,
    },
    "probability_columns": {
        "buy_col": "clf__prob_1",
        "short_col": None,
        "infer_short_from_buy": True,
    },
    "strategy": {
        "score_col": "buy_score_mean_raw_pct6",
        "component_threshold": 0.50,
        "top_k_values": [5, 10, 20, 40],
        "strategy_variants": ["raw_pct6", "momentum_21d"],
        "baseline_lookback_days": 21,
    },
    "synthetic_options": {
        "tenor_days": 60,
        "realized_vol_window": 21,
        "vol_floor": 0.15,
        "vol_cap": 0.80,
        "iv_multiplier": 1.0,
        "premium_floor": 0.25,
        "option_buckets": {
            "atm_option": {
                "long_strike_multiplier": 1.00,
                "short_strike_multiplier": 1.00,
            },
            "otm_option": {
                "long_strike_multiplier": 1.05,
                "short_strike_multiplier": 0.95,
            },
            "ditm_option": {
                "long_strike_multiplier": 0.90,
                "short_strike_multiplier": 1.10,
            },
        },
    },
}

display(pd.DataFrame([
    {
        "train_cutoff": APP_CFG["dates"]["train_cutoff"],
        "bt_window": f"{APP_CFG['dates']['bt_start']} to {APP_CFG['dates']['bt_end']}",
        "min_market_cap": APP_CFG["universe"]["min_market_cap"],
        "score_col": APP_CFG["strategy"]["score_col"],
        "component_threshold": APP_CFG["strategy"]["component_threshold"],
        "top_k_values": str(APP_CFG["strategy"]["top_k_values"]),
        "strategy_variants": str(APP_CFG["strategy"]["strategy_variants"]),
        "baseline_lookback_days": APP_CFG["strategy"]["baseline_lookback_days"],
        "option_tenor_days": APP_CFG["synthetic_options"]["tenor_days"],
        "option_buckets": str(APP_CFG["synthetic_options"]["option_buckets"]),
    }
]))


,train_cutoff,bt_window,min_market_cap,score_col,component_threshold,top_k_values,strategy_variants,baseline_lookback_days,option_tenor_days,option_buckets
0,2020-12-31,2021-01-01 to 2026-05-22,1.000000e+10,buy_score_mean_raw_pct6,0.5,"[5, 10, 20, 40]","['raw_pct6', 'momentum_21d']",21,60,"{'atm_option': {'long_strike_multiplier': 1.0,..."


In [ ]:
START_DATE = APP_CFG["dates"]["data_start"]
END_DATE = APP_CFG["dates"]["bt_end"]
TRAIN_CUTOFF_TS = pd.Timestamp(APP_CFG["dates"]["train_cutoff"])
BT_START_TS = pd.Timestamp(APP_CFG["dates"]["bt_start"])
BT_END_TS = pd.Timestamp(APP_CFG["dates"]["bt_end"])
SCORE_START_TS = pd.Timestamp(APP_CFG["dates"]["score_start"])

universe = tuple(
    resolve_symbol_universe(
        min_market_cap=float(APP_CFG["universe"]["min_market_cap"]),
        country=str(APP_CFG["universe"]["country"]),
        exchanges=list(APP_CFG["universe"]["exchanges"]),
        exclude_pooled_vehicles=bool(APP_CFG["universe"]["exclude_pooled_vehicles"]),
        limit=APP_CFG["universe"]["size"],
    )
)
if not universe:
    raise RuntimeError("No symbols resolved for the configured universe.")
log_memory("resolved universe", rows=len(universe), collect=True)

fmp_refresh_cfg = dict(APP_CFG.get("fmp_refresh", {}))
if bool(fmp_refresh_cfg.get("enabled", False)) and bool(resolve_fmp_api_key(required=False)):
    run_scoring_data_refresh_from_fmp(
        symbols=universe,
        target_start_date=START_DATE,
        target_end_date=END_DATE,
        refresh_mode=str(fmp_refresh_cfg.get("mode") or "scoring_ready"),
        refresh_symbol_sections_before_build=bool(fmp_refresh_cfg.get("refresh_symbol_sections_before_build", False)),
        refresh_macro_before_build=bool(fmp_refresh_cfg.get("refresh_macro_before_build", False)),
        max_symbols=fmp_refresh_cfg.get("max_symbols"),
        existing_historical_sections_only=bool(fmp_refresh_cfg.get("existing_historical_sections_only", True)),
        verbose=bool(fmp_refresh_cfg.get("verbose", True)),
        progress_logger=print,
    )

technical_df, technical_cols = _build_technical_dataframe_from_django(
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
)
if technical_df.empty:
    raise RuntimeError("No technical rows were built from Django prices.")
log_memory("built technical feature panel", rows=len(technical_df), cols=len(technical_df.columns), collect=True)

ctx = SimpleNamespace(api_key="")
macro_df, macro_cols = build_macro_dataframe(
    ctx=ctx,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    verbose=False,
)
log_memory("built macro feature panel", rows=len(macro_df), cols=len(macro_cols), collect=True)

# Build feature-family groups for classifier MoE.
# Price-derived technical features are grouped under the FMP adjusted-price endpoint.
fmp_endpoint_frames, fmp_endpoint_cols = _build_fmp_endpoint_feature_families(
    symbols=universe,
    target_index=technical_df.index,
    price_panel=technical_df,
    filing_lag_days=45,
)
log_memory("built FMP endpoint feature families", rows=sum(len(frame) for frame in fmp_endpoint_frames.values()), cols=sum(len(cols) for cols in fmp_endpoint_cols.values()), collect=True)
def _combine_feature_frames(frame_map, names):
    frames = [frame_map[name] for name in names if name in frame_map and frame_map[name] is not None and not frame_map[name].empty]
    if not frames:
        return pd.DataFrame(index=technical_df.index)
    combined = pd.concat(frames, axis=1).sort_index()
    if combined.columns.has_duplicates:
        combined = combined.loc[:, ~combined.columns.duplicated(keep="last")]
    numeric_cols = [c for c in combined.columns if pd.api.types.is_numeric_dtype(combined[c])]
    return combined[numeric_cols].copy()

macro_treasury_cols = [c for c in macro_cols if str(c).lower().startswith("ust") or "treasury" in str(c).lower()]
macro_economic_cols = [c for c in macro_cols if c not in macro_treasury_cols]
macro_endpoint_frames = {}
if macro_economic_cols:
    macro_endpoint_frames["macro_economic_indicators"] = macro_df[macro_economic_cols].copy()
if macro_treasury_cols:
    macro_endpoint_frames["macro_treasury_rates"] = macro_df[macro_treasury_cols].copy()

raw_feature_family_frames = {
    "prices_div_adj": technical_df[[c for c in technical_cols if c in technical_df.columns]].copy(),
    **fmp_endpoint_frames,
    **macro_endpoint_frames,
}

feature_family_groups = {
    "prices_div_adj": ["prices_div_adj"],
    "valuation_quality": ["key_metrics", "ratios"],
    "income_statement_bundle": ["income_statement", "income_statement_growth"],
    "balance_sheet_bundle": ["balance_sheet", "balance_sheet_growth"],
    "cash_flow_bundle": ["cash_flow", "cash_flow_growth"],
    "earnings_growth_bundle": ["earnings", "financial_growth"],
    "analyst_sentiment_bundle": ["analyst_estimates", "ratings_historical", "grades_historical"],
    "ownership_activity": ["insider_trading"],
    "macro_bundle": ["macro_economic_indicators", "macro_treasury_rates"],
}

feature_family_frames = {
    group_name: _combine_feature_frames(raw_feature_family_frames, endpoint_names)
    for group_name, endpoint_names in feature_family_groups.items()
}
feature_family_frames = {
    name: frame
    for name, frame in feature_family_frames.items()
    if frame is not None and not frame.empty and frame.shape[1] > 0
}
feature_family_weights = {name: 1.0 for name in feature_family_frames}
log_memory(
    "built initial feature family frames",
    rows=sum(len(frame) for frame in feature_family_frames.values()),
    cols=sum(frame.shape[1] for frame in feature_family_frames.values()),
    collect=True,
)


EXECUTION_PARAMS = {
    "price_col": "close",
    "fee_bps": float(APP_CFG["costs"]["fee_bps"]),
    "slippage_bps": float(APP_CFG["costs"]["slippage_bps"]),
}
WEIGHTING_PARAMS = {
    "use_sample_weight": bool(APP_CFG["labels"]["use_sample_weight"]),
    "alpha": float(APP_CFG["labels"]["alpha"]),
    "r_clip": float(APP_CFG["labels"]["r_clip"]),
    "horizon_balance": bool(APP_CFG["labels"]["horizon_balance"]),
}

if BT_START_TS <= TRAIN_CUTOFF_TS:
    raise ValueError(
        "Backtest start must be after train_cutoff so train/OOS labels do not overlap: "
        f"bt_start={BT_START_TS.date()} train_cutoff={TRAIN_CUTOFF_TS.date()}"
    )

print("Building oracle labels for the full available panel...")
daily_map_all = {
    s: technical_df.xs(s, level="symbol").loc[:BT_END_TS].copy()
    for s in universe
    if s in set(technical_df.index.get_level_values("symbol"))
}
label_df_all = build_label_dataframe(
    daily_by_symbol=daily_map_all,
    k_params=dict(APP_CFG["labels"]["k_params"]),
    execution_params=EXECUTION_PARAMS,
    weighting=WEIGHTING_PARAMS,
    add_rank_labels=True,
    verbose=False,
)
log_memory("built full label panel", rows=len(label_df_all), cols=len(label_df_all.columns), collect=True)
label_df_train = label_df_all.loc[
    label_df_all.index.get_level_values("date") <= TRAIN_CUTOFF_TS
].copy()
label_df_oos = label_df_all.loc[
    (label_df_all.index.get_level_values("date") >= BT_START_TS)
    & (label_df_all.index.get_level_values("date") <= BT_END_TS)
].copy()

print("Label/date split")
display(pd.DataFrame([
    {
        "train_cutoff": TRAIN_CUTOFF_TS.date().isoformat(),
        "bt_window": f"{BT_START_TS.date()} to {BT_END_TS.date()}",
        "all_label_rows": len(label_df_all),
        "train_label_rows": len(label_df_train),
        "oos_label_rows": len(label_df_oos),
        "feature_family_count": len(feature_family_frames),
        "feature_families": ", ".join(feature_family_frames.keys()),
        "feature_cols_by_family": str({name: int(frame.shape[1]) for name, frame in feature_family_frames.items()}),
        "ensemble_weights": str(feature_family_weights),
    }
]))


def _cap_training_rows(df, max_rows, *, random_state=1337):
    max_rows = int(max_rows or 0)
    if max_rows <= 0 or len(df) <= max_rows:
        return df
    weights = None
    if "sample_weight" in df.columns:
        weights = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(0.0)
        if float(weights.sum()) <= 0.0:
            weights = None
    return df.sample(
        n=max_rows,
        random_state=random_state,
        weights=weights,
        replace=weights is not None,
    ).sort_index()


def _sanitize_family_numeric_features(df, feature_cols=None, *, clip_abs=1e12):
    if df is None or df.empty:
        return pd.DataFrame(), []
    out = df.copy()
    cols = list(feature_cols or out.columns)
    cols = [c for c in cols if c in out.columns and pd.api.types.is_numeric_dtype(out[c])]
    if not cols:
        return out.iloc[0:0].copy(), []
    out.loc[:, cols] = out[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if clip_abs is not None and float(clip_abs) > 0.0:
        out.loc[:, cols] = out[cols].clip(lower=-float(clip_abs), upper=float(clip_abs))
    cols = [c for c in cols if out[c].notna().any()]
    if not cols:
        return out.iloc[0:0].copy(), []
    available_mask = out[cols].notna().any(axis=1)
    return out.loc[available_mask].copy(), cols


def _filter_family_available_rows(df, feature_cols=None):
    out, _cols = _sanitize_family_numeric_features(df, feature_cols)
    return out


def _downcast_numeric_frame(df, feature_cols=None):
    if df is None or df.empty:
        return df
    cols = list(feature_cols or df.columns)
    cols = [c for c in cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if not cols:
        return df
    out = df.copy()
    out.loc[:, cols] = out[cols].astype("float32")
    return out


def _build_feature_slice_from_families(*, start_ts=None, end_ts=None, cols_by_family=None):
    frames = []
    for family_name, family_df in feature_family_frames.items():
        frame = family_df
        if start_ts is not None:
            frame = frame.loc[frame.index.get_level_values("date") >= start_ts]
        if end_ts is not None:
            frame = frame.loc[frame.index.get_level_values("date") <= end_ts]
        if frame.empty:
            continue
        cols = list((cols_by_family or feature_list_by_family).get(family_name, []))
        cols = [col for col in cols if col in frame.columns]
        if not cols:
            continue
        frames.append(frame[cols])
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, axis=1).sort_index()
    if out.columns.has_duplicates:
        out = out.loc[:, ~out.columns.duplicated(keep="last")]
    return out


def _build_feature_slice_for_index(row_index, *, cols_by_family=None):
    if row_index is None or len(row_index) == 0:
        return pd.DataFrame(index=row_index)
    frames = []
    family_cols = cols_by_family or feature_list_by_family
    for family_name, family_df in feature_family_frames.items():
        cols = list(family_cols.get(family_name, []))
        cols = [col for col in cols if col in family_df.columns]
        if not cols:
            continue
        frames.append(family_df.reindex(row_index)[cols])
    if not frames:
        return pd.DataFrame(index=row_index)
    out = pd.concat(frames, axis=1)
    if out.columns.has_duplicates:
        out = out.loc[:, ~out.columns.duplicated(keep="last")]
    return out


def _family_coverage_summary(df, feature_cols=None):
    available = _filter_family_available_rows(df, feature_cols)
    if available.empty:
        return {"coverage_start": None, "coverage_end": None, "available_rows": 0}
    dates = pd.DatetimeIndex(pd.to_datetime(available.index.get_level_values("date")))
    return {
        "coverage_start": dates.min().date().isoformat(),
        "coverage_end": dates.max().date().isoformat(),
        "available_rows": int(len(available)),
    }


family_models = {}
family_training_rows = []
all_family_feature_cols = []

for family_name, family_df in feature_family_frames.items():
    if family_df.empty or family_df.shape[1] == 0:
        print(f"Skipping {family_name}: no feature columns.")
        continue

    family_coverage = _family_coverage_summary(family_df)
    family_features_train = family_df.loc[
        family_df.index.get_level_values("date") <= TRAIN_CUTOFF_TS
    ].copy()
    family_features_train = _filter_family_available_rows(family_features_train)
    if family_features_train.empty:
        print(f"Skipping {family_name}: no endpoint data available before train_cutoff.")
        continue

    family_train_df, family_feature_cols, _ = prepare_ml_dataset(
        features_df=family_features_train,
        labels_df=label_df_train,
        target_cols=["target", "trade_return"],
        weight_col="sample_weight",
        config=MLDatasetConfig(drop_nan_features=False),
        verbose=True,
    )
    if family_train_df.empty or not family_feature_cols:
        print(f"Skipping {family_name}: no train rows or active features after label join.")
        continue
    family_train_df, family_feature_cols = _sanitize_family_numeric_features(family_train_df, family_feature_cols)
    family_train_df = _downcast_numeric_frame(family_train_df, family_feature_cols)
    if family_train_df.empty or not family_feature_cols:
        print(f"Skipping {family_name}: no train rows with finite endpoint features after label join.")
        continue

    family_clf_train_df = _cap_training_rows(
        family_train_df,
        APP_CFG["runtime"]["max_clf_train_rows"],
        random_state=1337,
    )
    family_clf_train_df, family_feature_cols = _sanitize_family_numeric_features(family_clf_train_df, family_feature_cols)
    family_clf_train_df = _downcast_numeric_frame(family_clf_train_df, family_feature_cols)
    if family_clf_train_df.empty or not family_feature_cols:
        print(f"Skipping {family_name}: no finite rows remained after training cap.")
        continue

    log_memory(f"prepared {family_name} RF training data", rows=len(family_clf_train_df), cols=len(family_feature_cols), collect=True)
    print(f"Training {family_name} classifier expert with {len(family_feature_cols)} features...")
    spec_clf = FitSpec(
        feature_cols=list(family_feature_cols),
        target_col="target",
        weight_col="sample_weight",
        split_ratio=1.0,
        model_tag=f"classifier expert ({family_name}): predicts buy/positive class from target",
    )
    family_clf = SklearnRFClassifier(
        random_state=1337,
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
    )
    family_clf.fit(family_clf_train_df, spec_clf, verbose=True)
    log_memory(f"fit {family_name} RF classifier", rows=len(family_clf_train_df), cols=len(family_feature_cols), collect=True)

    family_reg = None
    family_reg_train_df = family_train_df.iloc[0:0].copy()
    reg_target_available = "trade_return" in family_train_df.columns
    if reg_target_available:
        reg_target = pd.to_numeric(family_train_df["trade_return"], errors="coerce")
        family_reg_source_df = family_train_df.loc[reg_target.notna()].copy()
        reg_max_rows = int(
            APP_CFG["runtime"].get(
                "max_reg_train_rows",
                APP_CFG["runtime"].get("max_clf_train_rows", 100000),
            )
        )
        family_reg_train_df = _cap_training_rows(
            family_reg_source_df,
            reg_max_rows,
            random_state=1337,
        )
        family_reg_train_df, reg_feature_cols = _sanitize_family_numeric_features(family_reg_train_df, family_feature_cols)
        family_reg_train_df = _downcast_numeric_frame(family_reg_train_df, reg_feature_cols)
        if not family_reg_train_df.empty and reg_feature_cols:
            log_memory(f"prepared {family_name} RF ranking regressor data", rows=len(family_reg_train_df), cols=len(reg_feature_cols), collect=True)
            print(f"Training {family_name} ranking regressor expert with {len(reg_feature_cols)} features...")
            spec_reg = FitSpec(
                feature_cols=list(reg_feature_cols),
                target_col="trade_return",
                weight_col="sample_weight",
                split_ratio=1.0,
                model_tag=f"ranking regressor expert ({family_name}): predicts realized trade_return for ranking",
            )
            family_reg = SklearnRFRegressor(
                test_size=0.0,
                random_state=1337,
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                max_features="sqrt",
                n_jobs=-1,
            )
            family_reg.fit(family_reg_train_df, spec_reg, verbose=True)
            log_memory(f"fit {family_name} RF ranking regressor", rows=len(family_reg_train_df), cols=len(reg_feature_cols), collect=True)
        else:
            print(f"Skipping {family_name} ranking regressor: no finite trade_return rows after training cap.")
    else:
        print(f"Skipping {family_name} ranking regressor: trade_return target is not available.")

    family_models[family_name] = {
        "feature_df": family_df,
        "feature_cols": list(family_feature_cols),
        "train_df": family_train_df,
        "clf_train_df": family_clf_train_df,
        "reg_train_df": family_reg_train_df,
        "clf": family_clf,
        "reg": family_reg,
        "coverage": family_coverage,
    }
    all_family_feature_cols.extend(str(c) for c in family_feature_cols)
    family_training_rows.append(
        {
            "family": family_name,
            "feature_cols": len(family_feature_cols),
            "train_rows": len(family_train_df),
            "clf_fit_rows": len(family_clf_train_df),
            "reg_fit_rows": len(family_reg_train_df) if family_reg is not None else 0,
            "available_rows_full_panel": int(family_coverage["available_rows"]),
            "coverage_start": family_coverage["coverage_start"],
            "coverage_end": family_coverage["coverage_end"],
            "weight": float(feature_family_weights.get(family_name, 1.0)),
        }
    )
    del family_features_train
    gc.collect()

if not family_models:
    raise RuntimeError("No classifier expert models were trained.")

family_training_df = pd.DataFrame(family_training_rows)
print("Classifier + trade-return ranking regressor expert training summary")
display(family_training_df)

primary_family_name = next((name for name in ("prices_div_adj", "valuation_quality", "macro_bundle") if name in family_models), next(iter(family_models)))
primary_family = family_models[primary_family_name]
clf_raw = primary_family["clf"]
raw_feature_list = sorted(set(all_family_feature_cols))
train_df = pd.concat([payload["train_df"] for payload in family_models.values()], keys=family_models.keys(), names=["family"])
clf_train_df = pd.concat([payload["clf_train_df"] for payload in family_models.values()], keys=family_models.keys(), names=["family"])
reg_train_payloads = {name: payload["reg_train_df"] for name, payload in family_models.items() if payload.get("reg") is not None}
reg_train_df = pd.concat(reg_train_payloads.values(), keys=reg_train_payloads.keys(), names=["family"]) if reg_train_payloads else pd.DataFrame()
model_source = "retrained_classifier_regressor_ae_moe"
artifact_status = {"age": pd.NaT, "reason": "forced_retrain"}

feature_list_by_family = {name: list(payload["feature_cols"]) for name, payload in family_models.items()}
ae_raw = None
ae_raw_numeric_cols = []
ae_train_df = pd.DataFrame()
ae_candidate_index = None
for payload in family_models.values():
    idx = payload["train_df"].index
    ae_candidate_index = idx if ae_candidate_index is None else ae_candidate_index.union(idx)

if ae_candidate_index is not None and len(ae_candidate_index) > 0:
    ae_index_df = pd.DataFrame(index=ae_candidate_index)
    if "sample_weight" in label_df_train.columns:
        ae_index_df = ae_index_df.join(label_df_train[["sample_weight"]], how="left")
    ae_max_rows = int(
        APP_CFG["runtime"].get(
            "max_ae_train_rows",
            APP_CFG["runtime"].get("max_clf_train_rows", 100000),
        )
    )
    ae_index_df = _cap_training_rows(ae_index_df, ae_max_rows, random_state=1337)
    ae_train_features = _build_feature_slice_for_index(ae_index_df.index, cols_by_family=feature_list_by_family)
    ae_train_features, ae_feature_cols = _sanitize_family_numeric_features(ae_train_features)
    ae_train_features = _downcast_numeric_frame(ae_train_features, ae_feature_cols)
    if not ae_train_features.empty and ae_feature_cols:
        ae_train_df = ae_train_features.copy()
        ae_train_df["label"] = 0
        log_memory("prepared autoencoder training data", rows=len(ae_train_df), cols=len(ae_feature_cols), collect=True)
        print(f"Training autoencoder familiarity model with {len(ae_feature_cols)} features...")
        ae_raw, ae_raw_numeric_cols = train_ae(ae_train_df, ae_feature_cols, verbose=True)
        log_memory("fit autoencoder familiarity model", rows=len(ae_train_df), cols=len(ae_raw_numeric_cols), collect=True)
    else:
        print("Skipping autoencoder: no finite combined feature rows after training cap.")
else:
    print("Skipping autoencoder: no family training rows were available.")

ae_predict_familiarity = make_autoencoder_familiarity_predictor(list(ae_raw_numeric_cols)) if ae_raw is not None else None

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score

INSAMPLE_START_TS = pd.Timestamp(START_DATE)
INSAMPLE_END_TS = TRAIN_CUTOFF_TS
OUTSAMPLE_START_TS = BT_START_TS
OUTSAMPLE_END_TS = BT_END_TS

print("Building aligned classifier/regressor expert diagnostics from fitted family models...")
print(f"  - In-sample:     {INSAMPLE_START_TS.date()} to {INSAMPLE_END_TS.date()}")
print(f"  - Out-of-sample: {OUTSAMPLE_START_TS.date()} to {OUTSAMPLE_END_TS.date()}")


def _predict_rf_classifier_probability(model, df):
    used_features = list(getattr(model, "_used_features", []))
    if not used_features:
        raise RuntimeError("Classifier does not expose fitted _used_features.")
    x = df[used_features].replace([np.inf, -np.inf], np.nan)
    raw_model = getattr(model, "model", model)
    proba = np.asarray(raw_model.predict_proba(x), dtype=float)
    classes = list(getattr(raw_model, "classes_", []))
    if 1 in classes:
        positive_idx = classes.index(1)
    elif "1" in classes:
        positive_idx = classes.index("1")
    else:
        positive_idx = proba.shape[1] - 1
    return proba[:, positive_idx]


def _predict_rf_regressor(model, df):
    used_features = list(getattr(model, "_used_features", []))
    if not used_features:
        raise RuntimeError("Regressor does not expose fitted _used_features.")
    x = df[used_features].replace([np.inf, -np.inf], np.nan)
    raw_model = getattr(model, "model", model)
    return np.asarray(raw_model.predict(x), dtype=float)


def _score_autoencoder_familiarity(row_index):
    if ae_raw is None or ae_predict_familiarity is None or not ae_raw_numeric_cols:
        return pd.Series(1.0, index=row_index, dtype=float)
    ae_features = _build_feature_slice_for_index(row_index, cols_by_family=feature_list_by_family)
    ae_features, ae_available_cols = _sanitize_family_numeric_features(ae_features, ae_raw_numeric_cols)
    if ae_features.empty:
        return pd.Series(0.0, index=row_index, dtype=float)
    for col in ae_raw_numeric_cols:
        if col not in ae_features.columns:
            ae_features[col] = np.nan
    ae_features = _downcast_numeric_frame(ae_features, ae_raw_numeric_cols)
    familiarity = pd.Series(
        ae_predict_familiarity(ae_features, ae_raw),
        index=ae_features.index,
        dtype=float,
    )
    return familiarity.reindex(row_index).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0.0, 1.0)


def _score_family_predictions(feature_df, family_payload, *, start_ts, end_ts, family_name):
    features = feature_df.loc[
        (feature_df.index.get_level_values("date") >= start_ts)
        & (feature_df.index.get_level_values("date") <= end_ts)
    ].copy()
    if features.empty:
        return pd.DataFrame(index=features.index)
    clf_features = list(getattr(family_payload["clf"], "_used_features", []))
    reg = family_payload.get("reg")
    reg_features = list(getattr(reg, "_used_features", [])) if reg is not None else []
    used_features = sorted(set(clf_features).union(reg_features))
    features, available_cols = _sanitize_family_numeric_features(features, used_features)
    if features.empty:
        return pd.DataFrame(index=features.index)
    for col in used_features:
        if col not in features.columns:
            features[col] = np.nan
    features = _downcast_numeric_frame(features, used_features)
    out = pd.DataFrame(index=features.index)
    out[f"{family_name}__prob_buy"] = _predict_rf_classifier_probability(family_payload["clf"], features)
    if reg is not None:
        out[f"{family_name}__ranking"] = _predict_rf_regressor(reg, features)
    return out


def _build_family_prediction_panel(*, start_ts, end_ts):
    frames = []
    active_weights = {}
    for family_name, payload in family_models.items():
        scored = _score_family_predictions(
            payload["feature_df"],
            payload,
            start_ts=start_ts,
            end_ts=end_ts,
            family_name=family_name,
        )
        if scored.empty:
            continue
        frames.append(scored)
        active_weights[family_name] = float(feature_family_weights.get(family_name, 1.0))
    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=1).sort_index()
    weight_sum = sum(active_weights.values())
    if weight_sum <= 0.0:
        raise RuntimeError("Classifier MoE weights must sum to a positive value.")

    weighted_sum = pd.Series(0.0, index=out.index, dtype=float)
    row_weight_sum = pd.Series(0.0, index=out.index, dtype=float)
    row_expert_count = pd.Series(0, index=out.index, dtype=int)
    ranking_weighted_sum = pd.Series(0.0, index=out.index, dtype=float)
    ranking_weight_sum = pd.Series(0.0, index=out.index, dtype=float)
    row_regressor_count = pd.Series(0, index=out.index, dtype=int)
    for family_name, weight in active_weights.items():
        prob = pd.to_numeric(out[f"{family_name}__prob_buy"], errors="coerce")
        valid = prob.notna()
        weighted_sum.loc[valid] += float(weight) * prob.loc[valid]
        row_weight_sum.loc[valid] += float(weight)
        row_expert_count.loc[valid] += 1
        ranking_col = f"{family_name}__ranking"
        if ranking_col in out.columns:
            ranking = pd.to_numeric(out[ranking_col], errors="coerce")
            valid_ranking = ranking.notna()
            ranking_weighted_sum.loc[valid_ranking] += float(weight) * ranking.loc[valid_ranking]
            ranking_weight_sum.loc[valid_ranking] += float(weight)
            row_regressor_count.loc[valid_ranking] += 1
    out["clf__prob_1"] = (weighted_sum / row_weight_sum.replace(0.0, np.nan)).clip(0.0, 1.0)
    out["clf"] = out["clf__prob_1"]
    out["moe_available_experts"] = row_expert_count
    out["ranking"] = ranking_weighted_sum / ranking_weight_sum.replace(0.0, np.nan)
    out["ranking"] = out["ranking"].fillna(out["clf__prob_1"])
    out["moe_available_regressors"] = row_regressor_count
    out["ae_familiarity"] = _score_autoencoder_familiarity(out.index)
    out["combined_score"] = out[["clf__prob_1", "ranking", "ae_familiarity"]].mean(axis=1)
    out["moe_active_experts"] = len(active_weights)
    log_memory(f"scored classifier MoE slice {start_ts.date()} to {end_ts.date()}", rows=len(out), cols=len(out.columns), collect=True)
    return out


def _build_direct_model_diagnostic_frame(split_name, labels_df, *, start_ts, end_ts):
    scored = _build_family_prediction_panel(start_ts=start_ts, end_ts=end_ts)
    labels = labels_df.loc[
        (labels_df.index.get_level_values("date") >= start_ts)
        & (labels_df.index.get_level_values("date") <= end_ts)
    ].copy()
    diagnostic_label_cols = [col for col in ["target", "trade_return", "sample_weight"] if col in labels.columns]
    out = scored.join(labels[diagnostic_label_cols], how="inner")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["clf__prob_1"])
    if out.empty:
        return out
    out["split"] = str(split_name)
    out["window_start"] = start_ts.date().isoformat()
    out["window_end"] = end_ts.date().isoformat()
    out["target_binary"] = (pd.to_numeric(out["target"], errors="coerce") > 0).astype(int)
    out["pred_binary"] = (pd.to_numeric(out["clf__prob_1"], errors="coerce") >= 0.50).astype(int)
    out["prob_buy"] = pd.to_numeric(out["clf__prob_1"], errors="coerce").fillna(0.0)
    out["diagnostic_year"] = out.index.get_level_values("date").year
    return out


label_df_insample_diagnostic = label_df_train.loc[
    (label_df_train.index.get_level_values("date") >= INSAMPLE_START_TS)
    & (label_df_train.index.get_level_values("date") <= INSAMPLE_END_TS)
].copy()
label_df_outsample_diagnostic = label_df_oos.loc[
    (label_df_oos.index.get_level_values("date") >= OUTSAMPLE_START_TS)
    & (label_df_oos.index.get_level_values("date") <= OUTSAMPLE_END_TS)
].copy()

model_diagnostic_frames = [
    _build_direct_model_diagnostic_frame(
        "in_sample_train_window",
        label_df_insample_diagnostic,
        start_ts=INSAMPLE_START_TS,
        end_ts=INSAMPLE_END_TS,
    ),
    _build_direct_model_diagnostic_frame(
        "out_of_sample_backtest_window",
        label_df_outsample_diagnostic,
        start_ts=OUTSAMPLE_START_TS,
        end_ts=OUTSAMPLE_END_TS,
    ),
]
model_diagnostic_frames = [frame for frame in model_diagnostic_frames if not frame.empty]
model_diagnostic_df = pd.concat(model_diagnostic_frames, axis=0).sort_index() if model_diagnostic_frames else pd.DataFrame()

if model_diagnostic_df.empty:
    raise RuntimeError("No rows with both classifier-MoE scores and realized labels were available for aligned diagnostics.")


def _safe_roc_auc(labels, scores):
    return float(roc_auc_score(labels, scores)) if pd.Series(labels).nunique(dropna=True) == 2 else np.nan


def _summarize_model_split(g):
    y_true = g["target_binary"].to_numpy(dtype=int)
    y_pred = g["pred_binary"].to_numpy(dtype=int)
    y_prob = g["prob_buy"].to_numpy(dtype=float)
    return pd.Series({
        "window": f"{g['window_start'].iloc[0]} to {g['window_end'].iloc[0]}",
        "rows": int(len(g)),
        "symbols": int(g.index.get_level_values("symbol").nunique()),
        "dates": int(g.index.get_level_values("date").nunique()),
        "positive_rate_actual": float(g["target_binary"].mean()),
        "positive_rate_predicted": float(g["pred_binary"].mean()),
        "accuracy_at_50pct": float(accuracy_score(y_true, y_pred)),
        "roc_auc_prob_buy": _safe_roc_auc(y_true, y_prob),
        "avg_actual_trade_return": float(pd.to_numeric(g["trade_return"], errors="coerce").mean()),
        "avg_predicted_trade_return": float(pd.to_numeric(g["ranking"], errors="coerce").mean()) if "ranking" in g.columns else np.nan,
    })


split_model_diagnostics = (
    model_diagnostic_df
    .groupby("split", sort=False)
    .apply(_summarize_model_split)
    .reset_index()
)

print("Aligned classifier-MoE diagnostics: train_cutoff vs backtest window")
display(split_model_diagnostics)

family_oos_diagnostics = []
for family_name in family_models:
    family_prob_col = f"{family_name}__prob_buy"
    for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
        if family_prob_col not in split_df.columns:
            continue
        y_true = split_df["target_binary"].to_numpy(dtype=int)
        y_prob = pd.to_numeric(split_df[family_prob_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        y_pred = (y_prob >= 0.50).astype(int)
        family_oos_diagnostics.append({
            "split": split_name,
            "family": family_name,
            "rows": int(len(split_df)),
            "accuracy_at_50pct": float(accuracy_score(y_true, y_pred)),
            "roc_auc_prob_buy": _safe_roc_auc(y_true, y_prob),
        })

print("Classifier expert diagnostics")
display(pd.DataFrame(family_oos_diagnostics))

trade_return_regressor_diagnostics = []
if "trade_return" in model_diagnostic_df.columns:
    for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
        y_true_all = pd.to_numeric(split_df["trade_return"], errors="coerce")
        ranking_cols = [("ensemble", "ranking")]
        ranking_cols.extend(
            (family_name, f"{family_name}__ranking")
            for family_name in family_models
            if f"{family_name}__ranking" in split_df.columns
        )
        for model_name, ranking_col in ranking_cols:
            y_pred_all = pd.to_numeric(split_df[ranking_col], errors="coerce")
            valid = y_true_all.notna() & y_pred_all.notna()
            if int(valid.sum()) < 2:
                continue
            y_true = y_true_all.loc[valid].to_numpy(dtype=float)
            y_pred = y_pred_all.loc[valid].to_numpy(dtype=float)
            trade_return_regressor_diagnostics.append({
                "split": split_name,
                "family": model_name,
                "rows": int(valid.sum()),
                "r2_trade_return": float(r2_score(y_true, y_pred)),
                "mae_trade_return": float(mean_absolute_error(y_true, y_pred)),
                "mse_trade_return": float(mean_squared_error(y_true, y_pred)),
                "actual_trade_return_mean": float(np.mean(y_true)),
                "predicted_trade_return_mean": float(np.mean(y_pred)),
            })

print("Trade-return ranking regressor diagnostics")
display(pd.DataFrame(trade_return_regressor_diagnostics))

for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
    print(f"Classification report: classifier MoE {split_name} | positive class = realized profitable/buy label")
    print(classification_report(
        split_df["target_binary"],
        split_df["pred_binary"],
        labels=[0, 1],
        target_names=["not_buy", "buy"],
        zero_division=0,
    ))
    print(f"Confusion matrix: classifier MoE {split_name}")
    display(pd.DataFrame(
        confusion_matrix(split_df["target_binary"], split_df["pred_binary"], labels=[0, 1]),
        index=["actual_not_buy", "actual_buy"],
        columns=["pred_not_buy", "pred_buy"],
    ))

yearly_model_diagnostics = (
    model_diagnostic_df
    .groupby(["split", "diagnostic_year"], sort=False)
    .apply(lambda g: pd.Series({
        "rows": int(len(g)),
        "positive_rate_actual": float(g["target_binary"].mean()),
        "positive_rate_predicted": float(g["pred_binary"].mean()),
        "accuracy_at_50pct": float(accuracy_score(g["target_binary"], g["pred_binary"])),
        "roc_auc_prob_buy": _safe_roc_auc(g["target_binary"], g["prob_buy"]),
        "avg_actual_trade_return": float(pd.to_numeric(g["trade_return"], errors="coerce").mean()),
        "avg_predicted_trade_return": float(pd.to_numeric(g["ranking"], errors="coerce").mean()) if "ranking" in g.columns else np.nan,
    }))
    .reset_index()
)

print("Yearly classifier-MoE diagnostics by aligned split")
display(yearly_model_diagnostics)

score_decile_diagnostics = []
for split_name, split_df in model_diagnostic_df.groupby("split", sort=False):
    decile_df = split_df.copy()
    q = min(10, max(1, int(decile_df["prob_buy"].notna().sum())))
    decile_df["prob_buy_decile"] = pd.qcut(
        decile_df["prob_buy"].rank(method="first"),
        q=q,
        labels=False,
        duplicates="drop",
    )
    decile_summary = (
        decile_df
        .groupby("prob_buy_decile", dropna=True)
        .agg(
            rows=("target_binary", "size"),
            mean_prob_buy=("prob_buy", "mean"),
            hit_rate=("target_binary", "mean"),
            avg_actual_trade_return=("trade_return", "mean"),
            avg_predicted_trade_return=("ranking", "mean") if "ranking" in decile_df.columns else ("prob_buy", "mean"),
        )
        .reset_index()
    )
    decile_summary.insert(0, "split", split_name)
    score_decile_diagnostics.append(decile_summary)

score_decile_diagnostics = pd.concat(score_decile_diagnostics, ignore_index=True) if score_decile_diagnostics else pd.DataFrame()

print("Probability decile diagnostics by aligned split")
display(score_decile_diagnostics)

feature_importance_rows = []
for family_name, payload in family_models.items():
    for feature, importance in sorted(payload["clf"].feature_importance().items(), key=lambda item: item[1], reverse=True)[:15]:
        feature_importance_rows.append({
            "family": family_name,
            "model": "classifier",
            "feature": feature,
            "importance": float(importance),
        })
    reg = payload.get("reg")
    if reg is not None:
        for feature, importance in sorted(reg.feature_importance().items(), key=lambda item: item[1], reverse=True)[:15]:
            feature_importance_rows.append({
                "family": family_name,
                "model": "trade_return_regressor",
                "feature": feature,
                "importance": float(importance),
            })
feature_importance_df = pd.DataFrame(feature_importance_rows)
print("Top classifier and trade-return regressor feature importances by expert")
display(feature_importance_df)

artifact_dir = Path(str(APP_CFG["runtime"]["artifact_dir"]))
artifact_dir.mkdir(parents=True, exist_ok=True)
with open(artifact_dir / "classifier_moe.pkl", "wb") as f:
    pickle.dump({name: payload["clf"] for name, payload in family_models.items()}, f)
with open(artifact_dir / "trade_return_regressor_moe.pkl", "wb") as f:
    pickle.dump({name: payload["reg"] for name, payload in family_models.items() if payload.get("reg") is not None}, f)
if ae_raw is not None:
    with open(artifact_dir / "autoencoder.pkl", "wb") as f:
        pickle.dump(ae_raw, f)
with open(artifact_dir / "classifier_moe_meta.json", "w") as f:
    json.dump(
        {
            "artifact_version": 1,
            "stack": "grouped_fmp_endpoint_classifier_trade_return_regressor_moe",
            "trained_families": list(family_models.keys()),
            "feature_family_weights": {name: float(feature_family_weights.get(name, 1.0)) for name in family_models},
            "feature_family_groups": {name: list(feature_family_groups.get(name, [])) for name in family_models},
            "feature_list_by_family": {name: list(payload["feature_cols"]) for name, payload in family_models.items()},
            "trade_return_regressor_families": [name for name, payload in family_models.items() if payload.get("reg") is not None],
            "has_ranking_regressor": False,
            "has_trade_return_regressor": any(payload.get("reg") is not None for payload in family_models.values()),
            "has_autoencoder": ae_raw is not None,
            "ae_numeric_cols": list(ae_raw_numeric_cols),
        },
        f,
        indent=2,
    )
saved_artifact_dir = artifact_dir

display(pd.DataFrame([
    {
        "universe_size": len(universe),
        "model_feature_source": "grouped FMP endpoint classifier + trade-return regressor MoE",
        "trained_families": ", ".join(family_models.keys()),
        "feature_panel_rows": max(len(payload["feature_df"]) for payload in family_models.values()),
        "feature_panel_cols": sum(len(payload["feature_cols"]) for payload in family_models.values()),
        "feature_family_count": len(family_models),
        "feature_cols_by_family": str({name: len(payload["feature_cols"]) for name, payload in family_models.items()}),
        "train_rows_total_across_families": int(family_training_df["train_rows"].sum()),
        "clf_fit_rows_total_across_families": int(family_training_df["clf_fit_rows"].sum()),
        "reg_fit_rows_total_across_families": int(family_training_df["reg_fit_rows"].sum()) if "reg_fit_rows" in family_training_df.columns else 0,
        "ae_fit_rows": int(len(ae_train_df)) if ae_raw is not None else 0,
        "ae_numeric_cols": int(len(ae_raw_numeric_cols)),
        "trade_return_regressor_families": ", ".join([name for name, payload in family_models.items() if payload.get("reg") is not None]),
        "artifact_dir": str(saved_artifact_dir),
        "model_source": model_source,
        "artifact_age_hours": None if pd.isna(artifact_status.get("age")) else round(float(artifact_status["age"] / pd.Timedelta(hours=1)), 2),
        "artifact_status": str(artifact_status.get("reason") or ""),
    }
]))


[memory] resolved universe: rss=525.0 MB | rows=762
[memory] built technical feature panel: rss=3,666.1 MB | rows=3,226,031 | cols=76
[memory] built macro feature panel: rss=4,134.3 MB | rows=3,226,031 | cols=17
[memory] built FMP endpoint feature families: rss=3,011.5 MB | rows=45,004,485 | cols=423
[memory] built initial feature family frames: rss=3,218.2 MB | rows=29,007,238 | cols=511
Building oracle labels for the full available panel...
[memory] built full label panel: rss=3,014.6 MB | rows=238,051 | cols=17
Label/date split


,train_cutoff,bt_window,all_label_rows,train_label_rows,oos_label_rows,feature_family_count,feature_families,feature_cols_by_family,ensemble_weights
0,2020-12-31,2021-01-01 to 2026-05-22,238051,163394,74657,9,"prices_div_adj, valuation_quality, income_stat...","{'prices_div_adj': 71, 'valuation_quality': 10...","{'prices_div_adj': 1.0, 'valuation_quality': 1..."


--- Preparing ML Training Dataset ---
  - Final Training Rows: 163,394
  - Active Features:     71
  - Targets:             ['target', 'trade_return']
  - Sample Weight Col:   sample_weight
[memory] prepared prices_div_adj RF training data: rss=3,918.2 MB | rows=163,394 | cols=71
Training prices_div_adj classifier expert with 71 features...

  DIAGNOSTIC: SKLEARN RANDOM FOREST CLASSIFIER (TEST RESULTS)
DATASET & SPLIT:
  - Total Observations: 163,394
  - Target:             target
  - Model Role:         classifier expert (prices_div_adj): predicts buy/positive class from target
  - Split Mode:         In-sample eval (no internal holdout split)
  - Features:           71 numeric (filtered 0 strings)
  - Positive Class:     1 => 1

CLASS DISTRIBUTION (Mapping: {0: '0', 1: '1'}):
               Train Set              Test Set
  -        0:  81,457 (49.9%)    81,457 (49.9%)
  -        1:  81,937 (50.1%)    81,937 (50.1%)

TEST PERFORMANCE:
  - Accuracy:           98.96%
  - ROC AUC:      

,family,feature_cols,train_rows,clf_fit_rows,reg_fit_rows,available_rows_full_panel,coverage_start,coverage_end,weight
0,prices_div_adj,71,163394,163394,163394,3226031,2005-01-03,2026-05-21,1.0
1,valuation_quality,101,161541,161541,161541,3199355,2005-01-03,2026-05-21,1.0
2,income_statement_bundle,66,161530,161530,161530,3199168,2005-01-03,2026-05-21,1.0
3,balance_sheet_bundle,109,161486,161486,161486,3198634,2005-01-03,2026-05-21,1.0
4,cash_flow_bundle,79,161511,161511,161511,3198912,2005-01-03,2026-05-21,1.0
5,earnings_growth_bundle,49,162607,162607,162607,3215072,2005-01-03,2026-05-21,1.0
6,analyst_sentiment_bundle,14,157332,157332,157332,3134804,2005-01-03,2026-05-21,1.0
7,ownership_activity,5,162381,162381,162381,3203658,2005-01-03,2026-05-21,1.0
8,macro_bundle,17,163394,163394,163394,3226031,2005-01-03,2026-05-21,1.0


[memory] prepared autoencoder training data: rss=2,852.5 MB | rows=100,000 | cols=511
Training autoencoder familiarity model with 511 features...

  DIAGNOSTIC: AUTOENCODER REPORT
### 1. MODEL STATE
- Type: NUMERIC_ONLY
- Numeric inputs: 511
- Categorical inputs: 0
- Encoder layers: 3
- Bottleneck dim: 63
- MSE (Mean): 18.078318
- MSE (P95):  43.841862

### 2. TOP FEATURE RECONSTRUCTION MSE (TRAIN, STANDARDIZED)
- bs__otherreceivables: 120.096556
- bsg__growthdeferredtaxliabilitiesnoncurrent: 99.780999
- bsg__growthcommonstock: 98.308765
- cfg__growthdebtrepayment: 97.375132
- cf__netpreferredstockissuance: 94.434481
- bsg__growthtreasurystock: 89.136998
- cfg__growthpurchasesofinvestments: 87.387254
- cfg__growthacquisitionsnet: 87.339007
- is__sellingandmarketingexpenses: 86.568757
- cf__interestpaid: 85.005868
[memory] fit autoencoder familiarity model: rss=950.4 MB | rows=100,000 | cols=511
Building aligned classifier/regressor expert diagnostics from fitted family models...
  - In

In [ ]:
prob_cfg = ProbabilityColumnConfig(**APP_CFG["probability_columns"])
scored_panel_all = _build_family_prediction_panel(
    start_ts=SCORE_START_TS,
    end_ts=BT_END_TS,
)
if scored_panel_all.empty:
    raise RuntimeError("No ensemble score rows were produced for the scoring panel.")

scored_panel_all = enrich_scored_panel(scored_panel_all, prob_config=prob_cfg)
bt_panel_all = make_backtest_panel(
    scored_panel=scored_panel_all,
    technical_df=technical_df,
    start=SCORE_START_TS,
    end=BT_END_TS,
)
bt_panel_5y = bt_panel_all.loc[
    (bt_panel_all.index.get_level_values("date") >= BT_START_TS)
    & (bt_panel_all.index.get_level_values("date") <= BT_END_TS)
].copy()

display(pd.DataFrame([
    {
        "scoring_rows": len(bt_panel_all),
        "oos_rows": len(bt_panel_5y),
        "oos_symbols": int(bt_panel_5y.index.get_level_values("symbol").nunique()),
        "oos_dates": int(bt_panel_5y.index.get_level_values("date").nunique()),
        "trained_families": ", ".join(family_models.keys()),
        "ensemble_weights": str(feature_family_weights),
        "score_source": "grouped FMP endpoint classifier + trade-return regressor + autoencoder",
    }
]))


In [ ]:
score_col = str(APP_CFG["strategy"]["score_col"])
short_score_col = resolve_short_score_col(score_col)
component_cols = resolve_component_cols(score_col)
short_component_cols = resolve_component_cols(short_score_col)
top_k_values = list(APP_CFG["strategy"]["top_k_values"])
strategy_variants = list(APP_CFG["strategy"].get("strategy_variants", ["raw_pct6"]))
years = list(range(BT_START_TS.year, BT_END_TS.year + 1))

base_inputs = _prepare_capacity_rule_inputs(bt_panel_5y, score_col, component_cols, "close")
close_panel = base_inputs["close"]
realized_vol_panel = build_realized_vol_panel(
    close_panel,
    window=int(APP_CFG["synthetic_options"]["realized_vol_window"]),
    vol_floor=float(APP_CFG["synthetic_options"]["vol_floor"]),
    vol_cap=float(APP_CFG["synthetic_options"]["vol_cap"]),
)

instrument_return_panels = {
    "equity": {
        "long": close_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
        "short": close_panel.pct_change().mul(-1.0).replace([np.inf, -np.inf], np.nan).fillna(0.0),
    },
}
synthetic_price_panels = {}
for option_name, option_bucket in APP_CFG["synthetic_options"]["option_buckets"].items():
    long_strike_multiplier = float(option_bucket["long_strike_multiplier"])
    short_strike_multiplier = float(option_bucket["short_strike_multiplier"])
    call_price_panel = build_constant_maturity_call_price_panel(
        close_panel,
        realized_vol_panel,
        strike_multiplier=long_strike_multiplier,
        tenor_days=int(APP_CFG["synthetic_options"]["tenor_days"]),
        iv_multiplier=float(APP_CFG["synthetic_options"]["iv_multiplier"]),
        premium_floor=float(APP_CFG["synthetic_options"]["premium_floor"]),
    )
    put_price_panel = build_constant_maturity_put_price_panel(
        close_panel,
        realized_vol_panel,
        strike_multiplier=short_strike_multiplier,
        tenor_days=int(APP_CFG["synthetic_options"]["tenor_days"]),
        iv_multiplier=float(APP_CFG["synthetic_options"]["iv_multiplier"]),
        premium_floor=float(APP_CFG["synthetic_options"]["premium_floor"]),
    )
    synthetic_price_panels[option_name] = {
        "call": call_price_panel,
        "put": put_price_panel,
        "long_strike_multiplier": long_strike_multiplier,
        "short_strike_multiplier": short_strike_multiplier,
    }
    instrument_return_panels[option_name] = {
        "long": call_price_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
        "short": put_price_panel.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0),
    }

variant_runs = {}
summary_rows = []
yearly_rows = []

strategy_specs = []
if "raw_pct6" in strategy_variants:
    strategy_specs.append(
        {
            "strategy": "raw_pct6_long_only",
            "kind": "ml_long_only",
            "score_col": score_col,
            "component_cols": component_cols,
            "component_threshold": float(APP_CFG["strategy"]["component_threshold"]),
        }
    )
    strategy_specs.append(
        {
            "strategy": "raw_pct6_long_short",
            "kind": "ml_long_short",
            "long_score_col": score_col,
            "short_score_col": short_score_col,
            "long_component_cols": component_cols,
            "short_component_cols": short_component_cols,
            "component_threshold": float(APP_CFG["strategy"]["component_threshold"]),
        }
    )
if "momentum_21d" in strategy_variants:
    strategy_specs.append(
        {
            "strategy": "momentum_21d",
            "kind": "momentum",
            "lookback_days": int(APP_CFG["strategy"].get("baseline_lookback_days", 21)),
        }
    )

for strategy_spec in strategy_specs:
    strategy_name = str(strategy_spec["strategy"])
    for top_k in top_k_values:
        if strategy_spec["kind"] == "ml_long_only":
            signal_run = run_top_k_long_only_score_rule(
                panel=bt_panel_5y,
                score_col=str(strategy_spec["score_col"]),
                component_cols=list(strategy_spec.get("component_cols", [])),
                component_threshold=float(strategy_spec.get("component_threshold", 0.0)),
                price_col="close",
                top_k=int(top_k),
                rebalance_freq=None,
            )
        elif strategy_spec["kind"] == "ml_long_short":
            signal_run = run_top_k_long_short_score_rule(
                panel=bt_panel_5y,
                long_score_col=str(strategy_spec["long_score_col"]),
                short_score_col=str(strategy_spec["short_score_col"]),
                long_component_cols=list(strategy_spec.get("long_component_cols", [])),
                short_component_cols=list(strategy_spec.get("short_component_cols", [])),
                component_threshold=float(strategy_spec.get("component_threshold", 0.0)),
                price_col="close",
                top_k=int(top_k),
                rebalance_freq=None,
            )
        else:
            signal_run = run_top_k_momentum_baseline(
                panel=bt_panel_5y,
                price_col="close",
                top_k=int(top_k),
                lookback_days=int(strategy_spec.get("lookback_days", 21)),
                rebalance_freq=None,
            )

        positions = signal_run["positions"]
        prior_positions = positions.shift(1).fillna(0)
        buy_count = int(((positions == 1) & (prior_positions != 1)).sum().sum())
        sell_count = int(((prior_positions == 1) & (positions != 1)).sum().sum())
        short_count = int(((positions == -1) & (prior_positions != -1)).sum().sum())
        cover_count = int(((prior_positions == -1) & (positions != -1)).sum().sum())
        avg_active_names = float(positions.ne(0).sum(axis=1).mean()) if len(positions) else np.nan
        avg_long_names = float((positions == 1).sum(axis=1).mean()) if len(positions) else np.nan
        avg_short_names = float((positions == -1).sum(axis=1).mean()) if len(positions) else np.nan

        for instrument_name, asset_return_spec in instrument_return_panels.items():
            bt = backtest_positions_with_directional_asset_returns(
                positions,
                asset_return_spec["long"],
                short_asset_returns=asset_return_spec["short"],
                initial_balance=100000.0,
                fee_bps=float(APP_CFG["costs"]["fee_bps"]),
                slippage_bps=float(APP_CFG["costs"]["slippage_bps"]),
            )
            mode = f"{strategy_name}_{instrument_name}_top_{top_k}"
            curve_summary = summarize_curve(bt["returns"], years, mode=mode)
            variant_runs[(strategy_name, instrument_name, int(top_k))] = {
                "positions": positions,
                "backtest": bt,
                "summary": curve_summary,
            }
            summary_rows.append(
                {
                    "strategy": strategy_name,
                    "instrument": instrument_name,
                    "top_k": int(top_k),
                    "total_return_pct": curve_summary["total_return_pct"],
                    "sharpe": curve_summary["sharpe"],
                    "max_drawdown_pct": curve_summary["max_drawdown_pct"],
                    "buy_count": buy_count,
                    "sell_count": sell_count,
                    "short_count": short_count,
                    "cover_count": cover_count,
                    "avg_active_names": avg_active_names,
                    "avg_long_names": avg_long_names,
                    "avg_short_names": avg_short_names,
                    "avg_turnover": float(bt["turnover"].mean()) if len(bt["turnover"]) else np.nan,
                }
            )
            yearly_df = curve_summary["yearly_df"].copy()
            yearly_df.insert(0, "strategy", strategy_name)
            yearly_df.insert(1, "instrument", instrument_name)
            yearly_df.insert(2, "top_k", int(top_k))
            yearly_rows.append(yearly_df)

summary_df = pd.DataFrame(summary_rows).sort_values(["strategy", "instrument", "top_k"]).reset_index(drop=True)
yearly_summary_df = pd.concat(yearly_rows, ignore_index=True)

print("Synthetic options comparison summary")
display(summary_df)

print("Yearly summary")
display(yearly_summary_df)


In [ ]:
summary_df.sort_values("sharpe", ascending=False)

In [ ]:
yearly_summary_df.groupby(["strategy", "instrument", "top_k"])[["total_return_pct", "max_drawdown_pct"]].describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

INSTRUMENT_NAMES = ["equity", "atm_option", "otm_option", "ditm_option"]
ML_STRATEGY_NAMES = ["raw_pct6_long_only", "raw_pct6_long_short"]


def get_growth_multiple(key):
    """
    Returns final growth multiple for a variant key:
    1.0 = flat
    2.0 = +100%
    0.5 = -50%
    """
    if key not in variant_runs:
        return float("-inf")

    eq = variant_runs[key]["summary"]["equity_curve"]

    if eq is None or len(eq) == 0:
        return float("-inf")

    base = max(float(eq.iloc[0]), 1e-12)
    final = float(eq.iloc[-1])

    return final / base


def get_total_return_pct(key):
    growth = get_growth_multiple(key)

    if not np.isfinite(growth):
        return float("-inf")

    return (growth - 1.0) * 100.0


def sort_keys_by_total_return(keys, reverse=True):
    return sorted(keys, key=get_total_return_pct, reverse=reverse)


def plot_variant_curves(
    variant_keys,
    *,
    title,
    label_fn=None,
    figsize=(12, 5),
    top_n=None,
):
    sorted_keys = sort_keys_by_total_return(variant_keys)

    if top_n is not None:
        sorted_keys = sorted_keys[:top_n]

    fig, ax = plt.subplots(figsize=figsize)
    plotted = 0

    for rank, key in enumerate(sorted_keys, start=1):
        if key not in variant_runs:
            continue

        eq = variant_runs[key]["summary"]["equity_curve"]

        if eq is None or len(eq) == 0:
            continue

        base = max(float(eq.iloc[0]), 1e-12)
        growth_curve = eq / base

        total_return_pct = get_total_return_pct(key)
        growth_multiple = get_growth_multiple(key)

        if label_fn is None:
            label = (
                f"#{rank} {key[0]} {key[1]} top_{key[2]} "
                f"| return={total_return_pct:,.1f}% "
                f"| {growth_multiple:,.2f}x"
            )
        else:
            label = label_fn(key, rank, total_return_pct, growth_multiple)

        growth_curve.plot(ax=ax, lw=2, label=label)
        plotted += 1

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Growth (start=1.0)")
    ax.grid(True, alpha=0.3)

    if plotted:
        ax.legend(title="Sorted by total return", fontsize=8)

    plt.show()


strategy_names = sorted(summary_df["strategy"].dropna().unique())
top_k_values = sorted(int(x) for x in summary_df["top_k"].dropna().unique())


# =========================
# Equity variants
# =========================

equity_keys = [
    (strategy_name, "equity", top_k)
    for strategy_name in strategy_names
    for top_k in top_k_values
]

plot_variant_curves(
    equity_keys,
    title="Equity Strategy Variants Sorted by Total Return",
    label_fn=lambda key, rank, ret, growth: (
        f"#{rank} {key[0]} equity top_{key[2]} "
        f"| return={ret:,.1f}% | {growth:,.2f}x"
    ),
)


# =========================
# OTM option variants
# =========================

otm_keys = [
    (strategy_name, "otm_option", top_k)
    for strategy_name in strategy_names
    for top_k in top_k_values
]

plot_variant_curves(
    otm_keys,
    title="OTM Option Strategy Variants Sorted by Total Return",
    label_fn=lambda key, rank, ret, growth: (
        f"#{rank} {key[0]} otm top_{key[2]} "
        f"| return={ret:,.1f}% | {growth:,.2f}x"
    ),
)


# =========================
# Optional: plot only top 5
# =========================

plot_variant_curves(
    otm_keys,
    title="Top 5 OTM Option Strategy Variants by Total Return",
    top_n=5,
)